[Home](../../README.md)

### Data Wrangling

This is a demonstration of data wrangling using [Pandas](https://pandas.pydata.org/) the library for data analysis and manipulation.

This Jupyter Notepad demonstrates different processes you can apply to your data to prepare it for feature engineering and model training. For this demonstration we will wrangle the diabetes data set you previewed in the last Jupyter Notebook.

> [!Note]
> None of these processes are destructive to the source CSV as long as you save the modified data to a new CSV.

#### Load the required dependencies

In [1]:
# Import frameworks
import pandas as pd

####  Store the data as a local variable

The data frame is a Pandas object that structures your tabular data into an appropriate format. It loads the complete data in memory so it is now ready for preprocessing.

In [2]:
data_frame = pd.read_csv("2.1.2.minecraft_100x100.csv")

#### Dealing with null values

Null values during data analysis can cause runtime errors and unexpected results. It is important to identify null values and deal with them appropriately before training a model.

The `isnull().sum()` method call returns the null values in any column.

In [3]:
pd.set_option("display.max_rows", None)
data_frame.isnull().sum()

chunk_x                                       0
chunk_z                                       0
dominant_biome                                0
minecraft:acacia_leaves                       0
minecraft:acacia_log                          0
minecraft:air                                 0
minecraft:amethyst_block                      0
minecraft:amethyst_cluster                    0
minecraft:andesite                            0
minecraft:azalea                              0
minecraft:azalea_leaves                       0
minecraft:azure_bluet                         0
minecraft:barrel                              0
minecraft:bedrock                             0
minecraft:bee_nest                            0
minecraft:beetroots                           0
minecraft:bell                                0
minecraft:big_dripleaf                        0
minecraft:big_dripleaf_stem                   0
minecraft:birch_fence                         0
minecraft:birch_leaves                  

If you have null data there are many ways to deal with the empty/null values. These are the two most common approaches.
1. Remove any row with a null value with a `dropna()` method call.
2. Replace missing values with another value with a `fillna()` method call. Generally, we use mean value for numerical columns because it may cause minimal changes in your mathematical analysis while maintaining the original size of the data.

Students should reflect why this example removes the null 'SEX' but replacing the mean 'Target'?

In [4]:
# No Null values

# data_frame = data_frame.dropna(subset=['SEX'])
# data_frame.isnull().sum()

In [5]:
# No Null values

# data_frame['Target'] = data_frame['Target'].fillna(data_frame['Target'].mean())
# data_frame.isnull().sum()

#### Replace data

In order to improve the readibility and usability of my targets, I am going to remove the common prefix from every value.

In [6]:
data_frame['dominant_biome'] = data_frame['dominant_biome'].apply(lambda x: x.replace('minecraft:', ''))
data_frame['dominant_biome'].head()

0    cold_ocean
1    cold_ocean
2    cold_ocean
3    cold_ocean
4    cold_ocean
Name: dominant_biome, dtype: str

Then, I'll do the same with my features (to improve load time or readability)

In [7]:
data_frame.columns = data_frame.columns.str.replace("minecraft:", "", regex=False)
data_frame.head()

,chunk_x,chunk_z,dominant_biome,acacia_leaves,acacia_log,air,amethyst_block,amethyst_cluster,andesite,azalea,...,white_carpet,white_concrete,white_stained_glass,white_terracotta,white_wool,wildflowers,yellow_bed,yellow_carpet,yellow_glazed_terracotta,yellow_wool
0,0,0,cold_ocean,0,0,66914,0,0,1086,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,cold_ocean,0,0,67433,0,0,1274,0,...,0,0,0,0,0,0,0,0,0,0
2,1,1,cold_ocean,0,0,67102,0,0,873,1,...,0,0,0,0,0,0,0,0,0,0
3,0,1,cold_ocean,0,0,67086,0,0,1174,10,...,0,0,0,0,0,0,0,0,0,0
4,-1,1,cold_ocean,0,0,66035,0,0,1003,0,...,0,0,0,0,0,0,0,0,0,0


#### Drop/combine unecessary targets

(for the purpose of simplifying the algorithm and preventing overfitting)

In [8]:
data_frame['dominant_biome'].unique()

<StringArray>
[              'cold_ocean',                    'beach',
                    'river',          'deep_cold_ocean',
                    'taiga',                   'forest',
                    'ocean',               'deep_ocean',
                   'plains',             'birch_forest',
              'dark_forest',  'old_growth_birch_forest',
 'windswept_gravelly_hills',    'old_growth_pine_taiga',
        'windswept_savanna',  'old_growth_spruce_taiga',
                    'swamp',                   'meadow']
Length: 18, dtype: str

In [9]:
data_frame['dominant_biome'] = data_frame['dominant_biome'].apply(
    lambda biome: 'ocean' if 'ocean' in biome else biome
)
data_frame["dominant_biome"] = data_frame["dominant_biome"].apply(
    lambda biome: "taiga" if "taiga" in biome else biome
)
data_frame["dominant_biome"] = data_frame["dominant_biome"].apply(
    lambda biome: "birch_forest" if "birch_forest" in biome else biome
)
data_frame["dominant_biome"].unique()

<StringArray>
[                   'ocean',                    'beach',
                    'river',                    'taiga',
                   'forest',                   'plains',
             'birch_forest',              'dark_forest',
 'windswept_gravelly_hills',        'windswept_savanna',
                    'swamp',                   'meadow']
Length: 12, dtype: str

#### Drop unecessary features

Many of my features are not necessary for the algorithms prediction, either being very rare (only in one or two chunks), or are in every single chunk no matter what (e.g. bedrock).

In [10]:
df = pd.read_csv("2.1.2.minecraft_100x100.csv")
zero_counts = (df == 0).sum(axis=0)
zero_summary = pd.DataFrame({"Column": df.columns, "Zero Count": zero_counts.values})
zero_summary = zero_summary.sort_values("Zero Count", ascending=True)
print(zero_summary.to_string(index=False))

                                    Column  Zero Count
                            dominant_biome           0
                       minecraft:deepslate           0
                          minecraft:gravel           0
                           minecraft:stone           0
                         minecraft:bedrock           0
                             minecraft:air           0
                      minecraft:copper_ore           3
                         minecraft:diorite           3
                            minecraft:tuff           3
                        minecraft:andesite           4
                         minecraft:granite           4
          minecraft:deepslate_redstone_ore           6
                        minecraft:iron_ore           7
           minecraft:deepslate_diamond_ore          23
              minecraft:deepslate_iron_ore          24
                            minecraft:dirt          29
              minecraft:deepslate_gold_ore          58
          

I have attempted to filter features by the amount of 0 values to determine their importance, but I have decided that I will need to manually select which features to keep based on my domain knowledge.

In [11]:
# chunk_x, chunk_z, dominant_biome, air, dirt, water, short_grass, grass_block, lava, sand, tall_seagrass, oak_leaves, birch_leaves, oak_log, tall_grass, birch_log, kelp, kelp_plant, oak_planks, oak_fence, dark_oak_log, dark_oak_leaves, spruce_leaves, spruce_log, bush, sandstone, brown_mushroom, dandelion, podzol, red_mushroom_block, mushroom_stem, poppy, coarse_dirt, oxeye_daisy, cornflower, azure_bluet, emerald_ore, lily_pad, barrel, sugar_cane, red_mushroom, lilac, lily_of_the_valley, rose_bush, peony, sweet_berry_bush, bee_nest, pumpkin, snow, acacia_leaves, acacia_log, blue_orchid

columns_to_keep = [
    "chunk_x",
    "chunk_z",
    "dominant_biome",
    "air",
    "dirt",
    "water",
    "short_grass",
    "grass_block",
    "lava",
    "sand",
    "tall_seagrass",
    "oak_leaves",
    "birch_leaves",
    "oak_log",
    "tall_grass",
    "birch_log",
    "kelp",
    "kelp_plant",
    "oak_planks",
    "oak_fence",
    "dark_oak_log",
    "dark_oak_leaves",
    "spruce_leaves",
    "spruce_log",
    "bush",
    "sandstone",
    "brown_mushroom",
    "dandelion",
    "podzol",
    "red_mushroom_block",
    "mushroom_stem",
    "poppy",
    "coarse_dirt",
    "oxeye_daisy",
    "cornflower",
    "azure_bluet",
    "emerald_ore",
    "lily_pad",
    "barrel",
    "sugar_cane",
    "red_mushroom",
    "lilac",
    "lily_of_the_valley",
    "rose_bush",
    "peony",
    "sweet_berry_bush",
    "bee_nest",
    "pumpkin",
    "snow",
    "acacia_leaves",
    "acacia_log",
    "blue_orchid",
]

data_frame = data_frame[columns_to_keep]
data_frame.head()

,chunk_x,chunk_z,dominant_biome,air,dirt,water,short_grass,grass_block,lava,sand,...,lily_of_the_valley,rose_bush,peony,sweet_berry_bush,bee_nest,pumpkin,snow,acacia_leaves,acacia_log,blue_orchid
0,0,0,ocean,66914,309,3509,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,ocean,67433,275,4174,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,1,ocean,67102,407,4202,6,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,1,ocean,67086,121,3825,34,0,0,96,...,0,0,0,0,0,0,0,0,0,0
4,-1,1,ocean,66035,400,3889,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0


#### Remove outliers

Outliers can skew your analysis on numerical columns, and it is important to remove them. We can use the 25th and 75th quartile on numerical data, to get the inter-quartile range. This allows us to estimate an acceptable range, and we can then filter out any values outside this range. Mathematically, outliers are values occurring outside 1.5 times the interquartile range (IQR) from the first quartile (Q1) or third quartile (Q3).

In [12]:
# get the inter-quartile range on all biomes blocks
# Outliers are negligible and may have to be adressed after cutting down features

blocks = [
    'air', 'water'
]
for block in blocks:
    print(data_frame[block].describe())
    Q1 = data_frame[block].quantile(0.10)
    Q3 = data_frame[block].quantile(0.90)
    IQR = Q3 - Q1
    print(f'Outliers are a {block} count above {Q3 + IQR} or below {Q1 - IQR}')

count    10201.000000
mean     66242.230075
std       3615.903324
min      49720.000000
25%      65083.000000
50%      66129.000000
75%      67392.000000
max      89631.000000
Name: air, dtype: float64
Outliers are a air count above 76104.0 or below 56589.0
count    10201.000000
mean      1811.044505
std       2248.280411
min          0.000000
25%         26.000000
50%        717.000000
75%       3223.000000
max      15724.000000
Name: water, dtype: float64
Outliers are a water count above 10104.0 or below -5052.0


In [13]:
# Outliers are not being filtered as they are negligible

# data_frame = data_frame[(data_frame['BP'] >= Q1 - 1.5 * IQR) & (data_frame['BP'] <= Q3 + 1.5 * IQR)]
# print(data_frame['BP'].describe())

#### Scaling features to a common range

Scaling the features makes it easier for machine learning algorithms to find the optimal solution, as the different scales of the features do not influence them.

In [14]:
# All features are on the exact same scale already, no need for feature scaling

# scale_feature = 'BP'
# #the minimum value with space for outliers
# MIN_BP = 55
# #the maximum value with space for outliers
# MAX_BP = 140
# #scale features
# data_frame[scale_feature] = [(X - MIN_BP) / (MAX_BP - MIN_BP) for X in data_frame[scale_feature]]
# data_frame.describe()

> [!important]
> You need to save the calculations for each dataset you scale for scaling new values for prediction. Use [2.1.2.data.records.md](2.1.2.data.records.md) to record this information.

#### Save the wrangled data to CSV

In [15]:
data_frame.to_csv('../2.2.Feature_Engineering/2.2.1.wrangled_data.csv', index=False)